In [ ]:
!pip install requests -q

import json
import pandas as pd
import requests
from pathlib import Path

# ============== CONFIG ==============
LIMIT = None          # only keep the first N modules -- set to None for all
OUTPUT_CSV = "mslearn_courses_dataset.csv"
OUTPUT_STRUCTURES_JSON = "mslearn_course_structures.json"
# ======================================

CATALOG_URL = "https://learn.microsoft.com/api/catalog/"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept": "application/json",
}

SCHEMA_ALIGNMENT_COLUMNS = [
    "generativeLayout", "saq", "selectAndReveal", "consent",
    "selectAndRevealHotSpots", "binaryList", "consult", "coreImage",
    "saqVideo", "popup", "podcast", "slideShow", "aiMentorInteraction",
]


def fetch_catalog_modules() -> list:
    """Fetches the ENTIRE modules catalog in one call -- no topic/keyword
    filter of any kind. This returns everything Microsoft Learn has."""
    print("Fetching full MS Learn catalog (modules)... this is a large response, may take a moment")
    resp = requests.get(CATALOG_URL, params={"type": "modules"}, headers=HEADERS, timeout=120)
    resp.raise_for_status()
    modules = resp.json().get("modules", [])
    print(f"Catalog loaded: {len(modules)} total modules")
    return modules


def fetch_all_units() -> dict:
    """One bulk call for unit titles/durations, keyed by unit uid."""
    print("Fetching unit metadata (titles/durations)...")
    resp = requests.get(CATALOG_URL, params={"type": "units"}, headers=HEADERS, timeout=120)
    resp.raise_for_status()
    units = resp.json().get("units", [])
    print(f"Unit metadata loaded: {len(units)} units")
    return {u["uid"]: u for u in units}


def classify_unit(unit_uid: str, unit_meta: dict) -> str:
    last_segment = unit_uid.rsplit(".", 1)[-1].lower()
    title = (unit_meta.get("title") or "").lower() if unit_meta else ""
    text = f"{last_segment} {title}"

    if "knowledge-check" in text or "quiz" in text:
        return "quiz"
    if "summary" in text:
        return "summary"
    if "exercise" in text or "lab" in text:
        return "assignment"
    if "video" in text:
        return "video"
    return "reading"


def get_category(module: dict) -> str:
    return "; ".join((module.get("subjects") or []) + (module.get("products") or []))


def get_depth_level(module: dict) -> str:
    levels = module.get("levels") or []
    return levels[0].title() if levels else "Not Specified"


def unit_display_name(unit_uid: str, unit_meta: dict) -> str:
    if unit_meta and unit_meta.get("title"):
        return unit_meta["title"]
    return unit_uid.rsplit(".", 1)[-1].replace("-", " ").title()


def build_module_row(module: dict, units_by_uid: dict):
    unit_uids = module.get("units") or []
    counts = {"video": 0, "reading": 0, "assignment": 0, "quiz": 0, "summary": 0}

    lectures, assessments, supplements = [], [], []

    for uid in unit_uids:
        meta = units_by_uid.get(uid, {})
        unit_type = classify_unit(uid, meta)
        counts[unit_type] += 1

        name = unit_display_name(uid, meta)
        duration = round(meta.get("duration_in_minutes", 0) or 0, 2)

        if unit_type == "quiz":
            assessments.append({"name": name, "type": "QUIZ", "duration_minutes": duration})
        elif unit_type == "assignment":
            assessments.append({"name": name, "type": "ASSIGNMENT", "duration_minutes": duration})
        elif unit_type == "summary":
            supplements.append({"name": name, "type": "SUPPLEMENT", "duration_minutes": duration})
        else:  # video, reading
            lectures.append({"name": name, "type": "LECTURE", "duration_minutes": duration})

    row = {
        "course_id": module.get("uid"),
        "provider": "Microsoft",
        "category": get_category(module),
        "duration_hours": round(module.get("duration_in_minutes", 0) / 60, 2),
        "num_chapters": module.get("number_of_children", len(unit_uids)),
        "depth_level": get_depth_level(module),
        "num_videos": counts["video"],
        "num_readings": counts["reading"],
        "num_assignments": counts["assignment"],
        "num_quizzes": counts["quiz"],
        "summary": counts["summary"],
        "url": module.get("url", ""),
    }
    for col in SCHEMA_ALIGNMENT_COLUMNS:
        row[col] = 0

    module_structure = {
        "course_id": row["course_id"],
        "url": row["url"],
        "modules": [
            {
                "module_name": module.get("title"),
                "description": module.get("summary", "") or "",
                "total_duration_minutes": module.get("duration_in_minutes", 0),
                "lectures": lectures,
                "assessments": assessments,
                "supplements": supplements,
            }
        ],
    }
    return row, module_structure


# --- run ---
all_modules = fetch_catalog_modules()
units_by_uid = fetch_all_units()

matched = all_modules  # no filtering -- take everything
if LIMIT:
    matched = matched[:LIMIT]
print(f"Building features for {len(matched)} modules (entire catalog, unfiltered)")

rows = []
structures = []
for i, module in enumerate(matched, start=1):
    try:
        row, structure = build_module_row(module, units_by_uid)
        rows.append(row)
        structures.append(structure)
        if i % 200 == 0 or i == len(matched):
            print(f"[{i}/{len(matched)}] processed...")
    except Exception as e:
        print(f"[{i}/{len(matched)}] FAILED on {module.get('uid')}: {e}")

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)
Path(OUTPUT_STRUCTURES_JSON).write_text(
    json.dumps(structures, indent=2, ensure_ascii=False), encoding="utf-8"
)

print(f"\nWrote {len(rows)} rows -> {OUTPUT_CSV}")
print(f"Wrote module structures -> {OUTPUT_STRUCTURES_JSON}")
display(df)

from google.colab import files
files.download(OUTPUT_CSV)
files.download(OUTPUT_STRUCTURES_JSON)

Fetching full MS Learn catalog (modules)... this is a large response, may take a moment
Catalog loaded: 3360 total modules
Fetching unit metadata (titles/durations)...
Unit metadata loaded: 27054 units
Building features for 3360 modules (entire catalog, unfiltered)
[200/3360] processed...
[400/3360] processed...
[600/3360] processed...
[800/3360] processed...
[1000/3360] processed...
[1200/3360] processed...
[1400/3360] processed...
[1600/3360] processed...
[1800/3360] processed...
[2000/3360] processed...
[2200/3360] processed...
[2400/3360] processed...
[2600/3360] processed...
[2800/3360] processed...
[3000/3360] processed...
[3200/3360] processed...
[3360/3360] processed...

Wrote 3360 rows -> mslearn_courses_dataset.csv
Wrote module structures -> mslearn_course_structures.json


,course_id,provider,category,duration_hours,num_chapters,depth_level,num_videos,num_readings,num_assignments,num_quizzes,...,consent,selectAndRevealHotSpots,binaryList,consult,coreImage,saqVideo,popup,podcast,slideShow,aiMentorInteraction
0,learn.wwl.experiment-azure-machine-learning,Microsoft,machine-learning; azure-machine-learning,1.08,10,Beginner,0,7,1,1,...,0,0,0,0,0,0,0,0,0,0
1,learn.dpu.recall-preview,Microsoft,windows-11,0.40,8,Beginner,0,6,0,1,...,0,0,0,0,0,0,0,0,0,0
2,learn.languages.csharp-if-elseif-else,Microsoft,app-development; dotnet; vs-code,0.93,7,Beginner,0,2,3,1,...,0,0,0,0,0,0,0,0,0,0
3,learn.wwl.get-started-with-dates-times-time-zones,Microsoft,dotnet,0.97,8,Beginner,0,5,1,1,...,0,0,0,0,0,0,0,0,0,0
4,learn.wwl.design-implement-azure-expressroute,Microsoft,data-engineering; networking; azure-expressrou...,0.95,11,Intermediate,0,8,2,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3355,learn.mec.empower-educators-explore-potential-...,Microsoft,microsoft-edge; office-powerpoint; ms-copilot;...,1.08,9,Beginner,0,7,0,1,...,0,0,0,0,0,0,0,0,0,0
3356,learn.mec.build-sel-skills-with-reflect,Microsoft,office-teams,0.67,8,Intermediate,0,6,0,1,...,0,0,0,0,0,0,0,0,0,0
3357,learn.mec.secure-k-12-school-it-practices-syst...,Microsoft,azure; windows,1.77,17,Intermediate,0,14,1,1,...,0,0,0,0,0,0,0,0,0,0
3358,learn.mec.cybersecurity-careers-minecraft-educ...,Microsoft,minecraft,0.53,7,Beginner,0,5,0,1,...,0,0,0,0,0,0,0,0,0,0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install requests -q

"""
Coursera — Automated Batch Scraper (catalog discovery + JSON-based detail)
================================================================================
Two-stage pipeline:

  1. DISCOVERY (reused from the earlier co.py) -- browses Coursera's public
     courses.v1 catalog API page by page and keeps courses whose
     name/description match SEARCH_QUERIES. Fast, gives course_id/slug/
     name/description/domainTypes.

  2. DETAIL -- for each matched course (up to TEST_LIMIT for this test run),
     fetches https://www.coursera.org/learn/{slug} with a plain requests.get()
     and reads window.__APOLLO_STATE__, the structured JSON Coursera's own
     frontend renders the page from.

     NOTE on what changed from co.py: the old script needed Playwright/
     headless Chromium because it assumed module data only exists after
     JavaScript runs. That assumption was wrong -- __APOLLO_STATE__ is
     already present in the raw HTML from a plain request. So this version
     has NO browser dependency at all, and is not fooled by markup/wording
     changes the way regex-on-rendered-text was.

Item -> content-type classification (same rules as coursera_course_features.py):
    name contains "quiz"    -> quiz     (checked first, overrides bucket)
    name contains "summary" -> summary  (checked second, overrides bucket)
    lectures bucket          -> video
    supplements bucket       -> reading
    assessments bucket       -> assignment (ASSIGNMENT / PROGRAMMING_ASSIGNMENT /
                                UNGRADED_LAB / APP_ITEM)

duration_hours logic:
    - material.totalDuration (ISO 8601) present -> converted to hours
    - else "self-paced" found in estimatedWorkload text -> "self-paced"
    - else "N months" found in estimatedWorkload text -> "N months"
    - else -> "unknown"

Output (this test run):
    courses_dataset_coursera.csv        -- one row per course, feature columns
    coursera_course_structures.json     -- {course_id: {module structure}}
    coursera_scrape_errors.json         -- courses that failed detail fetch, with reason

TEST_LIMIT is set to None for this run (all matched courses). Set it to a
number (e.g. 10) if you want to test on a smaller batch first.
"""

import re
import csv
import json
import time
from pathlib import Path

import requests

# ============== CONFIG ==============
SEARCH_QUERIES = [
    "machine learning",
    "data science",
    "artificial intelligence",
    "deep learning",
    "python",
]
TEST_LIMIT = None          # only scrape the first N matched courses -- set to None for all
DETAIL_DELAY_SECONDS = 1.5   # politeness delay between individual course-page fetches
DISCOVERY_DELAY_SECONDS = 0.3

OUTPUT_CSV = "courses_dataset_coursera.csv"
OUTPUT_STRUCTURES_JSON = "coursera_course_structures.json"
OUTPUT_ERRORS_JSON = "coursera_scrape_errors.json"
# ======================================

CATALOG_BASE = "https://api.coursera.org/api"
COURSE_FIELDS = "description,domainTypes,slug"
PAGE_SIZE = 200

API_HEADERS = {
    "User-Agent": "Mozilla/5.0 (data-collection script; contact: n/a)",
    "Accept": "application/json",
}
PAGE_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

ISO_DURATION_RE = re.compile(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?")
MONTHS_RE = re.compile(r"(\d+(?:\.\d+)?)\s*months?\b", re.IGNORECASE)

SCHEMA_ALIGNMENT_COLUMNS = [
    "generativeLayout", "saq", "selectAndReveal", "consent",
    "selectAndRevealHotSpots", "binaryList", "consult", "coreImage",
    "saqVideo", "popup", "podcast", "slideShow", "aiMentorInteraction",
]


# ---------------------------------------------------------------------------
# Stage 1: discovery via courses.v1 (reused from co.py, unchanged in logic)
# ---------------------------------------------------------------------------

def matches_query(course: dict, terms: list) -> set:
    haystack = f"{course.get('name', '')} {course.get('description', '')}".lower()
    return {t for t in terms if t.lower() in haystack}


def discover_courses(terms: list, session: requests.Session, limit=None):
    by_id = {}
    start = 0
    page = 0
    while True:
        params = {"start": start, "limit": PAGE_SIZE, "fields": COURSE_FIELDS}
        resp = session.get(f"{CATALOG_BASE}/courses.v1", params=params, headers=API_HEADERS, timeout=20)
        if resp.status_code != 200:
            print(f"[discovery] page {page} failed: HTTP {resp.status_code}")
            break
        elements = resp.json().get("elements", [])
        if not elements:
            break

        page_matches = 0
        for c in elements:
            if not c.get("slug"):
                continue
            hit_terms = matches_query(c, terms)
            if hit_terms:
                by_id[c["id"]] = c
                page_matches += 1
                if limit and len(by_id) >= limit:
                    print(f"[discovery] page {page}: {page_matches} matched (hit limit={limit})")
                    return by_id

        print(f"[discovery] page {page} (start={start}): {len(elements)} seen, {page_matches} matched (total kept {len(by_id)})")

        if len(elements) < PAGE_SIZE:
            break
        start += PAGE_SIZE
        page += 1
        time.sleep(DISCOVERY_DELAY_SECONDS)

    return by_id


# ---------------------------------------------------------------------------
# Stage 2: detail via __APOLLO_STATE__ (JSON-based, from coursera_course_features.py)
# ---------------------------------------------------------------------------

def fetch_html(url: str) -> str:
    resp = requests.get(url, headers=PAGE_HEADERS, timeout=20)
    resp.raise_for_status()
    return resp.text


def extract_apollo_state(html: str) -> dict:
    m = re.search(r"window\.__APOLLO_STATE__\s*=\s*(\{.*?\});\s*\n", html, re.DOTALL)
    if not m:
        raise ValueError("window.__APOLLO_STATE__ not found on page")
    return json.loads(m.group(1))


def iso_duration_to_hours(iso: str):
    if not iso:
        return None
    m = ISO_DURATION_RE.match(iso)
    if not m:
        return None
    h, mnt, s = (int(x) if x else 0 for x in m.groups())
    return round(h + mnt / 60 + s / 3600, 2)


def resolve_ref(state: dict, ref_obj: dict) -> dict:
    return state.get(ref_obj["__ref"], {})


def find_course_page(state: dict):
    for val in state.values():
        if isinstance(val, dict) and val.get("__typename") == "DescriptionPage_CoursePage":
            if val.get("material", {}).get("weeks"):
                return val
    return None


def classify_item(name: str, bucket: str) -> str:
    name_l = (name or "").lower()
    if "quiz" in name_l:
        return "quiz"
    if "summary" in name_l:
        return "summary"
    if bucket == "lectures":
        return "video"
    if bucket == "supplements":
        return "reading"
    return "assignment"


def resolve_classified_item(state: dict, ref_obj: dict, bucket: str) -> dict:
    item = resolve_ref(state, ref_obj)
    name = item.get("name")
    dur_h = iso_duration_to_hours(item.get("duration"))
    return {
        "name": name,
        "raw_type": item.get("typeName"),
        "type": classify_item(name, bucket),
        "duration_minutes": round(dur_h * 60, 1) if dur_h is not None else None,
    }


def resolve_item(state: dict, ref_obj: dict) -> dict:
    item = resolve_ref(state, ref_obj)
    dur_h = iso_duration_to_hours(item.get("duration"))
    return {
        "name": item.get("name"),
        "type": item.get("typeName"),
        "duration_minutes": round(dur_h * 60, 1) if dur_h is not None else None,
    }


def get_duration_hours(cp: dict):
    hours = iso_duration_to_hours(cp.get("material", {}).get("totalDuration"))
    if hours is not None:
        return hours
    workload_text = cp.get("estimatedWorkload") or ""
    if "self-paced" in workload_text.lower() or "self paced" in workload_text.lower():
        return "self-paced"
    months_match = MONTHS_RE.search(workload_text)
    if months_match:
        return f"{months_match.group(1)} months"
    return "unknown"


def get_category(cp: dict) -> str:
    parts = []
    for d in (cp.get("domains") or []):
        dn, sn = d.get("domainName"), d.get("subdomainName")
        if dn and sn:
            parts.append(f"{dn}: {sn}")
        elif dn:
            parts.append(dn)
    return "; ".join(parts)


def get_depth_level(cp: dict) -> str:
    mapping = {"BEGINNER": "Beginner", "INTERMEDIATE": "Intermediate",
               "ADVANCED": "Advanced", "MIXED": "Mixed"}
    return mapping.get(cp.get("difficultyLevel"), "Not Specified")


def build_course_row(url: str):
    """Returns (row_dict, modules_list). Raises on failure -- caller handles."""
    html = fetch_html(url)
    state = extract_apollo_state(html)
    cp = find_course_page(state)
    if not cp:
        raise ValueError("no module data (material.weeks) found on page")

    modules = []
    counts = {"video": 0, "reading": 0, "assignment": 0, "quiz": 0, "summary": 0}

    for week_ref in cp["material"]["weeks"]:
        week = resolve_ref(state, week_ref)
        for bucket in ("lectures", "assessments", "supplements"):
            for ref in (week.get(bucket) or []):
                counts[resolve_classified_item(state, ref, bucket)["type"]] += 1

        dur_h = iso_duration_to_hours(week.get("totalDuration"))
        modules.append({
            "module_name": week.get("name"),
            "description": week.get("description"),
            "total_duration_minutes": round(dur_h * 60, 1) if dur_h is not None else None,
            "lectures": [resolve_item(state, r) for r in (week.get("lectures") or [])],
            "assessments": [resolve_item(state, r) for r in (week.get("assessments") or [])],
            "supplements": [resolve_item(state, r) for r in (week.get("supplements") or [])],
        })

    row = {
        "course_id": cp.get("id"),
        "provider": "Coursera",
        "category": get_category(cp),
        "duration_hours": get_duration_hours(cp),
        "num_chapters": len(modules),
        "depth_level": get_depth_level(cp),
        "num_videos": counts["video"],
        "num_readings": counts["reading"],
        "num_assignments": counts["assignment"],
        "num_quizzes": counts["quiz"],
        "summary": counts["summary"],
        "url": url,
    }
    for col in SCHEMA_ALIGNMENT_COLUMNS:
        row[col] = 0

    return row, modules


# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------

print("=" * 60)
print(f"Step 1: Discovery (target: {TEST_LIMIT or 'all'} matched courses)")
print("=" * 60)
session = requests.Session()
courses_by_id = discover_courses(SEARCH_QUERIES, session, limit=TEST_LIMIT)
if not courses_by_id:
    raise RuntimeError("No courses matched -- check network access or broaden SEARCH_QUERIES.")

course_list = list(courses_by_id.values())
if TEST_LIMIT:
    course_list = course_list[:TEST_LIMIT]

print()
print("=" * 60)
print(f"Step 2: Fetching details for {len(course_list)} courses")
print("=" * 60)

rows = []
structures = {}
errors = []

for i, c in enumerate(course_list, start=1):
    slug = c.get("slug")
    url = f"https://www.coursera.org/learn/{slug}"
    print(f"[{i}/{len(course_list)}] {slug}")
    try:
        row, modules = build_course_row(url)
        rows.append(row)
        structures[row["course_id"]] = {"url": url, "modules": modules}
    except Exception as e:
        print(f"    FAILED: {e}")
        errors.append({"slug": slug, "url": url, "error": str(e)})
    time.sleep(DETAIL_DELAY_SECONDS)

print()
print("=" * 60)
print("Step 3: Writing output")
print("=" * 60)

if rows:
    fieldnames = (
        ["course_id", "provider", "category", "duration_hours", "num_chapters",
         "depth_level", "num_videos", "num_readings", "num_assignments",
         "num_quizzes", "summary", "url"]
        + SCHEMA_ALIGNMENT_COLUMNS
    )
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"Wrote {len(rows)} rows -> {OUTPUT_CSV}")

Path(OUTPUT_STRUCTURES_JSON).write_text(
    json.dumps(structures, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"Wrote module structures -> {OUTPUT_STRUCTURES_JSON}")

if errors:
    Path(OUTPUT_ERRORS_JSON).write_text(
        json.dumps(errors, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print(f"{len(errors)} course(s) failed -- see {OUTPUT_ERRORS_JSON}")

print()
print(f"Done. {len(rows)} succeeded, {len(errors)} failed, out of {len(course_list)} attempted.")

if rows:
    import pandas as pd
    display(pd.DataFrame(rows))

# Download outputs directly from Colab
from google.colab import files
files.download(OUTPUT_CSV)
files.download(OUTPUT_STRUCTURES_JSON)
if errors:
    files.download(OUTPUT_ERRORS_JSON)

Step 1: Discovery (target: all matched courses)
[discovery] page 0 (start=0): 200 seen, 27 matched (total kept 27)
[discovery] page 1 (start=200): 200 seen, 23 matched (total kept 50)
[discovery] page 2 (start=400): 200 seen, 27 matched (total kept 77)
[discovery] page 3 (start=600): 200 seen, 30 matched (total kept 107)
[discovery] page 4 (start=800): 200 seen, 26 matched (total kept 133)
[discovery] page 5 (start=1000): 200 seen, 36 matched (total kept 169)
[discovery] page 6 (start=1200): 200 seen, 36 matched (total kept 205)
[discovery] page 7 (start=1400): 200 seen, 25 matched (total kept 230)
[discovery] page 8 (start=1600): 200 seen, 30 matched (total kept 260)
[discovery] page 9 (start=1800): 200 seen, 19 matched (total kept 279)
[discovery] page 10 (start=2000): 200 seen, 27 matched (total kept 306)
[discovery] page 11 (start=2200): 200 seen, 22 matched (total kept 328)
[discovery] page 12 (start=2400): 200 seen, 22 matched (total kept 350)
[discovery] page 13 (start=2600): 20

,course_id,provider,category,duration_hours,num_chapters,depth_level,num_videos,num_readings,num_assignments,num_quizzes,...,consent,selectAndRevealHotSpots,binaryList,consult,coreImage,saqVideo,popup,podcast,slideShow,aiMentorInteraction
0,B6cXoZ3VEe6J0A5c28jyYQ,Coursera,Data Science: Machine Learning; Computer Scien...,2.32,1,Beginner,10,9,11,1,...,0,0,0,0,0,0,0,0,0,0
1,YnC4qHR6Ee2hYgoC8bKiUw,Coursera,Data Science: Machine Learning; Information Te...,17.79,10,Intermediate,35,13,9,4,...,0,0,0,0,0,0,0,0,0,0
2,iGYdtQA7EfGQkhLpkq4N-Q,Coursera,Information Technology: Data Management; Data ...,1.28,2,Intermediate,3,2,4,0,...,0,0,0,0,0,0,0,0,0,0
3,spneOVkrEfGesw6FbrgxOQ,Coursera,Computer Science: Software Development; Inform...,6.94,7,Intermediate,45,12,6,0,...,0,0,0,0,0,0,0,0,0,0
4,3K3rBiYYEe6JJAruShrzEQ,Coursera,Information Technology: Cloud Computing,0.32,1,Beginner,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3070,0EM6vbZqEe-JHhL4HU-TUQ,Coursera,Computer Science: Algorithms; Information Tech...,5.44,10,Intermediate,27,1,9,0,...,0,0,0,0,0,0,0,0,0,0
3071,rhI6rXEFEe-CUA7-5BJkaw,Coursera,Information Technology: Cloud Computing,3.97,1,Intermediate,5,1,3,0,...,0,0,0,0,0,0,0,0,0,0
3072,QdzgZduyEe6fcwr_8ZRNUw,Coursera,Information Technology: Support and Operations...,18.46,7,Beginner,12,13,13,8,...,0,0,0,0,0,0,0,0,0,0
3073,tJApe-HgEeeSKxKWkG_JMg,Coursera,Data Science: Probability and Statistics; Math...,12.45,6,Advanced,18,8,5,0,...,0,0,0,0,0,0,0,0,0,0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# --- Colab cell: merge two CSVs with the same columns ---
import pandas as pd
from google.colab import files

# Upload the two CSV files
print("Upload the first CSV:")
uploaded1 = files.upload()
file1 = list(uploaded1.keys())[0]

print("Upload the second CSV:")
uploaded2 = files.upload()
file2 = list(uploaded2.keys())[0]

# Read both CSVs
df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# Merge (stack rows, since columns are the same)
merged_df = pd.concat([df1, df2], ignore_index=True)

# Save and download the merged CSV
merged_df.to_csv("merged.csv", index=False)
files.download("merged.csv")

print(f"Merged shape: {merged_df.shape}")
merged_df.head()

Upload the first CSV:


Saving courses_dataset_coursera.csv to courses_dataset_coursera (1).csv
Upload the second CSV:


Saving mslearn_courses_dataset.csv to mslearn_courses_dataset (1).csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Merged shape: (6435, 25)


,course_id,provider,category,duration_hours,num_chapters,depth_level,num_videos,num_readings,num_assignments,num_quizzes,...,consent,selectAndRevealHotSpots,binaryList,consult,coreImage,saqVideo,popup,podcast,slideShow,aiMentorInteraction
0,B6cXoZ3VEe6J0A5c28jyYQ,Coursera,Data Science: Machine Learning; Computer Scien...,2.32,1,Beginner,10,9,11,1,...,0,0,0,0,0,0,0,0,0,0
1,YnC4qHR6Ee2hYgoC8bKiUw,Coursera,Data Science: Machine Learning; Information Te...,17.79,10,Intermediate,35,13,9,4,...,0,0,0,0,0,0,0,0,0,0
2,iGYdtQA7EfGQkhLpkq4N-Q,Coursera,Information Technology: Data Management; Data ...,1.28,2,Intermediate,3,2,4,0,...,0,0,0,0,0,0,0,0,0,0
3,spneOVkrEfGesw6FbrgxOQ,Coursera,Computer Science: Software Development; Inform...,6.94,7,Intermediate,45,12,6,0,...,0,0,0,0,0,0,0,0,0,0
4,3K3rBiYYEe6JJAruShrzEQ,Coursera,Information Technology: Cloud Computing,0.32,1,Beginner,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# ── Colab cell: normalize course CSV (updated schema) ──
import json, os
import numpy as np
import pandas as pd
from google.colab import files

# ─────────────────────────────────────────────
print("Upload your CSV to normalize:")
uploaded = files.upload()
INPUT_CSV = list(uploaded.keys())[0]
OUTPUT_CSV = "normalized_output.csv"
base = os.path.splitext(OUTPUT_CSV)[0]
# ─────────────────────────────────────────────

# 'summary' removed from here — it's now normalized like a count column, not a flag
CUSTOM_CONTENT_FEATURES = [
    "generativeLayout", "saq", "selectAndReveal", "consent",
    "selectAndRevealHotSpots", "binaryList", "consult", "coreImage",
    "saqVideo", "popup", "podcast", "slideShow", "aiMentorInteraction",
]

# count-type columns to normalize the same way (divide-by-max)
# 'summary' added here per your instruction
COUNT_COLUMNS = ["num_chapters", "num_videos", "num_readings",
                  "num_assignments", "num_quizzes", "summary"]

DEPTH_MAP = {"beginner": 0, "intermediate": 1, "advanced": 2, "not specified": -1}
BOOL_MAP = {True: 1, False: 0, "True": 1, "False": 0, "true": 1, "false": 0,
            1: 1, 0: 0, "1": 1, "0": 0}


class SortedLabelEncoder:
    def __init__(self): self.mapping = {}
    def fit(self, s):
        labels = sorted(set(str(v).strip() for v in s.dropna()
                             if str(v).strip() not in ("", "nan")))
        self.mapping = {l: i + 1 for i, l in enumerate(labels)}
        return self
    def transform(self, s):
        return s.apply(lambda v: self.mapping.get(str(v).strip(), 0))
    def save(self, path):
        json.dump({"mapping": self.mapping}, open(path, "w"), indent=2)


class CategoryEncoder:
    """Splits on ';' (matches this dataset's 'A: B; C: D' format)."""
    def __init__(self): self.mapping = {}
    def _tokenize(self, cell):
        if pd.isna(cell) or str(cell).strip() in ("", "nan"):
            return []
        return [t.strip() for t in str(cell).split(";") if t.strip()]
    def fit(self, s):
        toks = set()
        for c in s: toks.update(self._tokenize(c))
        self.mapping = {t: i + 1 for i, t in enumerate(sorted(toks))}
        return self
    def transform(self, s):
        def enc(cell):
            toks = self._tokenize(cell)
            return "0" if not toks else ",".join(str(self.mapping.get(t, 0)) for t in toks)
        return s.apply(enc)
    def save(self, path):
        json.dump({"mapping": self.mapping}, open(path, "w"), indent=2)


def normalize(input_path, output_path):
    print(f"Reading: {input_path}")
    df = pd.read_csv(input_path)
    print(f"Original: {df.shape[0]} rows × {df.shape[1]} columns\n")

    out = pd.DataFrame(index=df.index)

    # ── provider ─────────────────────────────
    prov_enc = SortedLabelEncoder().fit(df["provider"])
    out["provider_encoded"] = prov_enc.transform(df["provider"])
    print(f"provider → {len(prov_enc.mapping)} labels")

    # ── category ─────────────────────────────
    cat_enc = CategoryEncoder().fit(df["category"])
    out["category"] = cat_enc.transform(df["category"])
    print(f"category → {len(cat_enc.mapping)} unique tokens")

    # ── depth_level ──────────────────────────
    out["depth_level"] = (
        df["depth_level"].astype(str).str.strip().str.lower()
        .map(DEPTH_MAP).fillna(-1).astype(int)
    )

    # ── count columns: divide-by-max (num_chapters, num_videos, ..., summary) ──
    count_stats = {}
    for col in COUNT_COLUMNS:
        vals = pd.to_numeric(df[col], errors="coerce")
        vals = vals.fillna(vals.median())
        cmax = float(vals.max()) if vals.max() > 0 else 1.0
        out[f"{col}_norm"] = (vals / cmax).round(6)
        count_stats[col] = cmax
        print(f"{col}_norm → divide by max ({cmax:.0f})  "
              f"range [{out[f'{col}_norm'].min():.4f}, {out[f'{col}_norm'].max():.4f}]")

    # ── duration_hours: min-max + is_self_paced flag ──
    hours = pd.to_numeric(df["duration_hours"], errors="coerce")
    is_self_paced = hours.isna().astype(int)
    known = hours.dropna()
    dmin = float(known.min()) if len(known) else 0.0
    dmax = float(known.max()) if len(known) else 1.0
    span = dmax - dmin
    norm = (hours - dmin) / span if span > 0 else hours.apply(lambda v: 0.0)
    norm = norm.clip(0.0, 1.0).fillna(0.0).round(6)
    out["duration_hours_norm"] = norm
    out["is_self_paced"] = is_self_paced
    print(f"duration_hours_norm → min-max [{dmin:.2f}, {dmax:.2f}]  "
          f"self_paced rows: {int(is_self_paced.sum())}")

    # ── reserved content-type flags: read real values if present, else 0 ──
    for col in CUSTOM_CONTENT_FEATURES:
        if col in df.columns:
            out[col] = df[col].map(BOOL_MAP).fillna(0).astype(int)
        else:
            out[col] = 0
    print(f"content flags → read {sum(c in df.columns for c in CUSTOM_CONTENT_FEATURES)}"
          f"/{len(CUSTOM_CONTENT_FEATURES)} from source, rest defaulted to 0")

    # ── assemble output: course_id + url passthrough + all normalized features ──
    passthrough_cols = ["course_id"]
    if "url" in df.columns:
        passthrough_cols.append("url")

    df_out = pd.concat([df[passthrough_cols].reset_index(drop=True),
                         out.reset_index(drop=True)], axis=1)
    df_out.to_csv(output_path, index=False)

    prov_enc.save(base + "_provider_map.json")
    cat_enc.save(base + "_category_map.json")
    json.dump({"count_max": count_stats, "duration_min": dmin, "duration_max": dmax},
               open(base + "_meta.json", "w"), indent=2)

    print(f"\nOutput: {output_path}  →  {df_out.shape[0]} rows × {df_out.shape[1]} cols")
    return df_out


df_out = normalize(INPUT_CSV, OUTPUT_CSV)
df_out.head()

Upload your CSV to normalize:


Saving merged.csv to merged (1).csv
Reading: merged (1).csv
Original: 6435 rows × 25 columns

provider → 2 labels
category → 387 unique tokens
num_chapters_norm → divide by max (51)  range [0.0196, 1.0000]
num_videos_norm → divide by max (246)  range [0.0000, 1.0000]
num_readings_norm → divide by max (271)  range [0.0000, 1.0000]
num_assignments_norm → divide by max (157)  range [0.0000, 1.0000]
num_quizzes_norm → divide by max (134)  range [0.0000, 1.0000]
summary_norm → divide by max (23)  range [0.0000, 1.0000]
duration_hours_norm → min-max [0.12, 114.25]  self_paced rows: 0
content flags → read 13/13 from source, rest defaulted to 0

Output: normalized_output.csv  →  6435 rows × 26 cols


,course_id,url,provider_encoded,category,depth_level,num_chapters_norm,num_videos_norm,num_readings_norm,num_assignments_norm,num_quizzes_norm,...,consent,selectAndRevealHotSpots,binaryList,consult,coreImage,saqVideo,popup,podcast,slideShow,aiMentorInteraction
0,B6cXoZ3VEe6J0A5c28jyYQ,https://www.coursera.org/learn/developing-ai-p...,1,"16,14",0,0.019608,0.040650,0.03321,0.070064,0.007463,...,0,0,0,0,0,0,0,0,0,0
1,YnC4qHR6Ee2hYgoC8bKiUw,https://www.coursera.org/learn/machine-learnin...,1,"16,27",1,0.196078,0.142276,0.04797,0.057325,0.029851,...,0,0,0,0,0,0,0,0,0,0
2,iGYdtQA7EfGQkhLpkq4N-Q,https://www.coursera.org/learn/unlock-multimod...,1,"28,16",1,0.039216,0.012195,0.00738,0.025478,0.000000,...,0,0,0,0,0,0,0,0,0,0
3,spneOVkrEfGesw6FbrgxOQ,https://www.coursera.org/learn/ai-assisted-cod...,1,"14,28",1,0.137255,0.182927,0.04428,0.038217,0.000000,...,0,0,0,0,0,0,0,0,0,0
4,3K3rBiYYEe6JJAruShrzEQ,https://www.coursera.org/learn/introduction-to...,1,27,0,0.019608,0.004065,0.00000,0.000000,0.007463,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# ── separate cell: download results ──
files.download(OUTPUT_CSV)
files.download(base + "_provider_map.json")
files.download(base + "_category_map.json")
files.download(base + "_meta.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>